# Multi-Modal, Training-Free Crack Extraction *via* Generalized Frangi Graph


This notebook contains the experiments reported in our accepted EUVIP 2026 paper:
1.  the **[FIND benchmark](https://doi.org/10.5281/zenodo.6383044)** (clean data, modality ablation, and synthetic-noise experiments); and
2.  the **Palais des Papes** intensity/depth case study.

**See the readme of the GitHub repository for more details**

---

📄 If you use this code for your research, please cite:

```bibtex
@inproceedings{HauseuxEUVIP2026,
  title     = {Multi-Modal, Training-Free Crack Extraction via Generalized Frangi Graph},
  author    = {Hauseux, Louis and Antoine, Raphaël and Foucher, Philippe and Charbonnier, Pierre and Zerubia, Josiane},
  booktitle = {European Workshop on Visual Information Processing (EUVIP)},
  year      = {2026},
  note      = {Accepted}
}
```


# Reproducibility: Google Drive Setup

To fully reproduce the experiments (especially the comparison with CrackSegDiff), you need to set up your Google Drive with a specific folder structure.

1.  **Mount Drive**: The notebook will mount your drive to `/content/drive`.
2.  **Required Path**: Create the following directory structure:
    `/content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff/20000_1000/test_output_fused/`
    Place the CrackSegDiff result images (masks) in this folder. The filenames should match the pattern `imXXXXX_output_ens.png`.
3.  **For Noisy Experiments**: If you run the noise benchmark, the notebook expects (or generates) noisy results in:
    `/content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff_noise/`

If you do not have the CrackSegDiff results, the comparison cells will simply skip those metrics, but the rest of the pipeline (our method) will still run.

The non-public geological data and precomputed comparison outputs are available in the [companion data folder](https://drive.google.com/drive/folders/1iC7QrdB2vZmaunjjPrIcu9r5wMeBPMlk).


In [ ]:
# 1. Environment Setup
!pip -q install numpy scipy scikit-image matplotlib joblib tqdm tqdm-joblib hdbscan networkx gdown tifffile imageio pandas Pillow pot

import os
if not os.path.exists("Frangi-EUVIP"):
    !git clone https://github.com/Ayana-Inria/Frangi-EUVIP.git

%cd Frangi-EUVIP
!pip install -e .


In [ ]:
import os, sys, numpy as np, matplotlib.pyplot as plt
import gdown, imageio.v2 as iio, pandas as pd
from pathlib import Path
from scipy.io import loadmat
import h5py
from skimage import io, color, img_as_float32
from skimage.transform import resize
from skimage.morphology import binary_closing, binary_opening, disk
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
from joblib import Parallel, delayed
from PIL import Image

# Ensure local src is importable
if os.path.exists("src"):
    sys.path.append(os.path.abspath("src"))
elif os.path.exists("Frangi-EUVIP/src"):
    sys.path.append(os.path.abspath("Frangi-EUVIP/src"))

from frangi_fusion import (
    set_seed, load_modalities_and_gt_by_index, to_gray,
    compute_hessians_per_scale, fuse_hessians_per_scale,
    build_frangi_similarity_graph, distances_from_similarity, triangle_connectivity_graph,
    largest_connected_component, hdbscan_from_sparse,
    mst_on_cluster, extract_backbone_centrality, skeleton_from_mst_graph,
    skeletonize_lee, thicken, jaccard_index, tversky_index, wasserstein_distance_skeletons,
    auto_discover_find_structure
)

# --- Global Hyper-parameters ---
Σ = [1, 3, 5, 7] #, 9]  # Gaussian scales
β = 2 # 0.5    # Frangi blob sensitivity
c = 0.25   # Contrast sensitivity
c_θ = 0.125 # Orientation sensitivity
R = 12      # Graph neighbor radius
K = 1      # 1 = Standard, 2 = Triangle Connectivity
threshold_mask = 0.6
dark_ridges = True

# expZ = 1
# min_cluster_size = 1500
# max_dist = R
# min_samples = 3
# allow_single_cluster = True

f_threshold = 0.35
min_centrality = 0.10

print("Libraries loaded and parameters defined.")


## Methodology: Hessian Fusion and Generalized Frangi Graph

### 1. Multi-Modal Hessian Fusion
For each modality $m$ (e.g., Intensity, Depth) and scale $\sigma \in \Sigma$, we compute the normalized Hessian matrix $\hat{\mathcal{H}}_{\sigma}^{(m)}(\mathbf{x})$ at pixel $\mathbf{x}$. To handle disparate dynamic ranges, we normalize by the maximum spectral norm:
$$ \hat{\mathcal{H}}_{\sigma}^{(m)}(\mathbf{x}) = \frac{\mathcal{H}_{\sigma}^{(m)}(\mathbf{x})}{\max_{\mathbf{y} \in \Omega} \| \mathcal{H}_{\sigma}^{(m)}(\mathbf{y}) \|} $$
The **Fused Hessian** is a weighted linear combination:
$$ \mathcal{H}_{\sigma}^{\text{fused}}(\mathbf{x}) = \sum_{m} w_m \hat{\mathcal{H}}_{\sigma}^{(m)}(\mathbf{x}) $$
We analyze eigenvalues $\lambda_1, \lambda_2$ (with $|\lambda_1| \le |\lambda_2|$) and eigenvectors $\mathbf{v}_1, \mathbf{v}_2$. For cracks (dark ridges), we expect $\lambda_2 > 0$.

### 2. Generalized Frangi Similarity
We construct a graph where edges connect pixels $i, j$ within the configured radius $R$. The pairwise similarity $S_{ij} ∈ [0,1]$ enforces local tubular geometry:

1.  **Elongation ($S_{\text{shape}}$):** Using ratio $\mathcal{R}_B = |\lambda_1| / |\lambda_2|$:
$$ S_{\text{shape}} = \exp\left(-\frac{1}{2} \left(\frac{\mathcal{R}_B(\mathbf{x}_i) + \mathcal{R}_B(\mathbf{x}_j)}{s_{\mathrm{s}}}\right)^2\right) \quad (\text{Note: } s_{\mathrm{s}} \text{ corresponds to } \beta \text{ in code}) $$
2.  **Contrast ($S_{\text{int}}$):** Using energy $\mathcal{S} = \|\mathcal{H}\|$:
$$ S_{\text{int}} = 1 - \exp\left(-\frac{1}{2} \left(\frac{\sqrt{\mathcal{S}(\mathbf{x}_i) \cdot \mathcal{S}(\mathbf{x}_j)}}{s_{\mathrm{i}}}\right)^2\right) \quad (\text{Note: } s_{\mathrm{i}} \text{ corresponds to } c \text{ in code}) $$
3.  **Alignment ($S_{\text{align}}$):** Penalizing deviation from $\mathbf{v}_1$:
$$ S_{\text{align}} = \exp\left(-\frac{1}{2} \left(\frac{\sin(\delta_\theta)}{s_{\mathrm{a}}}\right)^2\right) \quad (\text{Note: } s_{\mathrm{a}} \text{ corresponds to } c_\theta \text{ in code}) $$

After the spatial correction used in the implementation, the final dissimilarity is $d_{ij}=1-S_{ij}$.

### 3. Topological Extraction
The graph is reduced to its largest connected component and a **minimum spanning tree**. A similarity-weighted tree centrality then extracts the final skeleton.


## Part 1: Case Study - Palais des Papes (Avignon)
We demonstrate the power of fusion on a challenging real-world case: the retaining rock of the Palais des Papes.


In [ ]:
# Download Avignon Data
avignon_dir = "data_avignon"
os.makedirs(avignon_dir, exist_ok=True)
ortho_id = "1OXK3XNdrirwvnwI5yZI2AKlh3qSiSeDd"
mne_id = "1iB6RllC9augWlAshqE2e08vnglHHM_PQ"
ortho_path = os.path.join(avignon_dir, "Ortho_new_extrait.tif")
mne_path = os.path.join(avignon_dir, "MNE_new_extrait.mat")

if not os.path.exists(ortho_path):
    gdown.download(id=ortho_id, output=ortho_path, quiet=False)
if not os.path.exists(mne_path):
    gdown.download(id=mne_id, output=mne_path, quiet=False)
print("Avignon data downloaded.")


In [ ]:
# Helper function to load .mat files
def load_mat_2d(path, key=None, dtype=np.float32, squeeze=True):
    path = Path(path)
    try:
        from scipy.io import loadmat
        mdict = loadmat(path)
        user_keys = [k for k in mdict.keys() if not k.startswith('__')]
        if key is None: key = user_keys[0]
        arr = mdict[key]
    except NotImplementedError:
        import h5py
        with h5py.File(path, "r") as f:
            available = list(f.keys())
            if key is None: key = available[0]
            arr = np.array(f[key])
    if squeeze: arr = np.squeeze(arr)
    return np.asarray(arr, dtype=dtype)

# Load and Preprocess
IMG_PATH = ortho_path
img = io.imread(IMG_PATH)
if img.ndim == 2: img_gray = img_as_float32(img)
else: img_gray = img_as_float32(color.rgb2gray(img))

Scale_Z = 100.0
F = 5 # Downsampling factor for speed
depth_raw = load_mat_2d(mne_path, key="mne")
depth_map = Scale_Z / F * depth_raw

# Resize
if img_gray.shape != depth_map.shape:
    depth_map = resize(depth_map, img_gray.shape, order=1, preserve_range=True)
new_shape = (img_gray.shape[0] // F, img_gray.shape[1] // F)
img_gray_small = resize(img_gray, new_shape, order=1, anti_aliasing=True, preserve_range=True)
depth_map_small = resize(depth_map, new_shape, order=1, anti_aliasing=True, preserve_range=True)

print(f"Processed shapes: Image {img_gray_small.shape}, Depth {depth_map_small.shape}")

# Visualize Inputs
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img_gray_small, cmap='gray'); ax[0].set_title("Intensity"); ax[0].axis("off")
ax[1].imshow(img_gray_small, cmap='gray')
im = ax[1].imshow(depth_map_small, cmap='seismic', alpha=0.6)
ax[1].set_title("Intensity + Depth"); ax[1].axis("off")
# plt.colorbar(im, ax=ax[1])
plt.show()

In [ ]:
def run_pipeline(img_input, modality_name, custom_hessian=None):
    # 1. Hessian
    if custom_hessian is None:
        hessians = compute_hessians_per_scale(img_input, Σ)
        mods = {modality_name: hessians}
        weights = {modality_name: 1.0}
        fused_H = fuse_hessians_per_scale(mods, weights)
    else:
        fused_H = custom_hessian

    # 2. Graph
    coords, _, S = build_frangi_similarity_graph(fused_H, β, c, c_θ, R, candidate_mask=None, threshold_mask=threshold_mask, dark_ridges=dark_ridges)
    D = distances_from_similarity(S, mode="minus")
    if K == 2: D = triangle_connectivity_graph(coords, D)

    # 3. Extraction
    D_cc, idx_nodes = largest_connected_component(D)
    mask = np.zeros_like(img_input)
    if D_cc.shape[0] > 0:
        Dist = D_cc.copy(); Dist.setdiag(0.0)
        # labels = hdbscan_from_sparse(Dist, min_cluster_size=min_cluster_size, allow_single_cluster=True)
        # --- MODIFICATION: SKIP HDBSCAN ---
        labels = np.zeros(D_cc.shape[0], dtype=int)
        # ----------------------------------
        sub_coords = coords[idx_nodes]
        all_edges = []
        for lab in np.unique(labels):
            if lab < 0: continue
            cl = np.where(labels == lab)[0]
            if cl.size < 3: continue
            mst = mst_on_cluster(D_cc, cl)
            global_indices = idx_nodes[cl]
            S_cluster = S[global_indices, :][:, global_indices]
            nodes_kept, skel_graph = extract_backbone_centrality(mst, f_threshold=f_threshold, S=S_cluster, take_similarity=True, min_centrality=min_centrality)
            segs = skeleton_from_mst_graph(skel_graph, sub_coords[cl], nodes_kept, S=S_cluster, take_similarity=True)
            if segs.shape[0] > 0: all_edges.append(segs)
        if all_edges:
            fault_edges = np.vstack(all_edges)
            for e in fault_edges:
                r0, c0, r1, c1, _ = e
                rr, cc = np.linspace(r0, r1, int(max(abs(r1-r0), abs(c1-c0))+1)), np.linspace(c0, c1, int(max(abs(r1-r0), abs(c1-c0))+1))
                rr, cc = np.clip(rr.astype(int), 0, mask.shape[0]-1), np.clip(cc.astype(int), 0, mask.shape[1]-1)
                mask[rr, cc] = 1.0
    return fused_H, mask


In [ ]:
# --- Avignon Tests: Intensity, Range, Fused (2/3 - 1/3) ---
import matplotlib.pyplot as plt
import numpy as np
from scipy.sparse.csgraph import breadth_first_order

# Define configurations
configs = [
    ("Intensity Only", {"intensity": 1.0, "depth": 0.0}),
    ("Range Only",     {"intensity": 0.0, "depth": 1.0}),
    ("Fused (0.67/0.33)", {"intensity": 2/3, "depth": 1/3})
]

# Pre-compute Hessians (to save time)
print("Computing Hessians...")
# Ensure we work with the variables from previous cells
if 'img_gray_small' not in locals():
    print("Error: img_gray_small not found. Run previous cells.")
else:
    hess_int = compute_hessians_per_scale(img_gray_small, Σ)
    hess_depth = compute_hessians_per_scale(depth_map_small, Σ)
    mods_avail = {"intensity": hess_int, "depth": hess_depth}

    # Prepare figure: 3 rows (configs), 4 columns (|λ2|, Similarity, Centrality, Result)
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))

    for i, (name, weights) in enumerate(configs):
        print(f"Running {name}...")

        # 1. Fuse
        fused_H = fuse_hessians_per_scale(mods_avail, weights)

        # 2. Pipeline Steps (Graph & Extraction)
        # Custom pipeline run to get Similarity
        coords, _, S = build_frangi_similarity_graph(fused_H, β, c, c_θ, R, candidate_mask=None, threshold_mask=threshold_mask, dark_ridges=dark_ridges)
        D = distances_from_similarity(S, mode="minus")
        if K == 2: D = triangle_connectivity_graph(coords, D)

        D_cc, idx_nodes = largest_connected_component(D)
        mask = np.zeros_like(img_gray_small)
        centrality_vis = np.zeros_like(img_gray_small) # Map for visualization

        if D_cc.shape[0] > 0:
            # Skip HDBSCAN, use single cluster logic or simple CC
            labels = np.zeros(D_cc.shape[0], dtype=int)

            sub_coords = coords[idx_nodes]
            all_edges = []
            for lab in np.unique(labels):
                if lab < 0: continue
                cl = np.where(labels == lab)[0]
                if cl.size < 3: continue
                mst = mst_on_cluster(D_cc, cl)
                global_indices = idx_nodes[cl]
                S_cluster = S[global_indices, :][:, global_indices]

                # --- Compute Centrality for Visu ---
                N_cl = mst.shape[0]
                order_bfs, predecessors = breadth_first_order(mst, i_start=0, directed=False, return_predecessors=True)
                node_weights = np.ones(N_cl, dtype=np.float64)
                if S_cluster is not None:
                     node_weights = S_cluster.max(axis=1).toarray().flatten().astype(np.float64)
                subtree_mass = node_weights.copy()
                for idx_bfs in order_bfs[::-1]:
                    if idx_bfs != 0:
                        p = predecessors[idx_bfs]
                        if p>=0 and p<N_cl: subtree_mass[p] += subtree_mass[idx_bfs]
                total_mass = subtree_mass[0]
                cent = subtree_mass * (total_mass - subtree_mass)
                if cent.max() > 0: cent /= cent.max()

                # Map back
                rows, cols = sub_coords[cl][:, 0], sub_coords[cl][:, 1]
                centrality_vis[rows, cols] = cent
                # -----------------------------------

                nodes_kept, skel_graph = extract_backbone_centrality(mst, f_threshold=f_threshold, S=S_cluster, take_similarity=True, min_centrality=min_centrality)
                segs = skeleton_from_mst_graph(skel_graph, sub_coords[cl], nodes_kept, S=S_cluster, take_similarity=True)
                if segs.shape[0] > 0: all_edges.append(segs)

            if all_edges:
                fault_edges = np.vstack(all_edges)
                for e in fault_edges:
                    r0, c0, r1, c1, _ = e
                    rr, cc = np.linspace(r0, r1, int(max(abs(r1-r0), abs(c1-c0))+1)), np.linspace(c0, c1, int(max(abs(r1-r0), abs(c1-c0))+1))
                    rr, cc = np.clip(rr.astype(int), 0, mask.shape[0]-1), np.clip(cc.astype(int), 0, mask.shape[1]-1)
                    mask[rr, cc] = 1.0

        # 3. Visualization Data
        # A. Fused |lambda2|
        l2_stack = np.stack([Hd['e2n'] for Hd in fused_H], axis=0)
        best_idx = np.abs(l2_stack).argmax(axis=0)
        H, W = l2_stack.shape[1], l2_stack.shape[2]
        yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
        l2_vis = l2_stack[best_idx, yy, xx]

        # B. Similarity
        sim_vis = np.zeros_like(img_gray_small)
        if S.shape[0] > 0:
            degs = np.array(S.max(axis=1).toarray()).flatten()
            sim_vis[coords[:,0], coords[:,1]] = degs

        # Plotting
        # Column 0: Fused |Lambda2|
        ax0 = axes[i, 0]
        ax0.imshow(np.abs(l2_vis), cmap='magma')
        ax0.set_title(f"{name} - |λ2|")
        ax0.axis('off')

        # Column 1: Similarity
        ax1 = axes[i, 1]
        ax1.imshow(sim_vis, cmap='inferno')
        ax1.set_title(f"{name} - Similarity")
        ax1.axis('off')

        # Column 2: Centrality
        ax2 = axes[i, 2]
        ax2.imshow(centrality_vis, cmap='inferno')
        ax2.set_title(f"{name} - Centrality")
        ax2.axis('off')

        # Column 3: Result Overlay
        ax3 = axes[i, 3]
        ax3.imshow(img_gray_small, cmap='gray')
        ax3.imshow(mask, cmap='Reds', alpha=0.5)
        ax3.set_title(f"{name} - Result")
        ax3.axis('off')

    plt.tight_layout()
    plt.show()

## Part 2: FIND Dataset Benchmark


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
# Download FIND Dataset if not present
import zipfile
url = "https://drive.google.com/uc?id=1qnLMCeon7LJjT9H0ENiNF5sFs-F7-NvK"
zip_path = "data.zip"
extract_dir = "data_find"
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    if not os.path.exists(zip_path):
        gdown.download(url, zip_path, quiet=False)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    print("Unzipped FIND dataset.")
else:
    print("FIND dataset already present.")


In [ ]:
# --- Single Example Analysis (Seed=1) ---
import hdbscan

seed = 1

print(f"Analyzing image index {seed}...")
struct = auto_discover_find_structure("data_find")
dat = load_modalities_and_gt_by_index(struct, seed)
base = dat["arrays"].get("intensity", next(iter(dat["arrays"].values())))
gt = (dat["arrays"].get("label", np.zeros_like(base)) > 0).astype(np.uint8)
gt = binary_closing(gt, footprint=disk(2))
gt = binary_opening(gt, footprint=disk(2))

# 1. Compute Hessians & Fusion
mods_hess = {}
weights = {"intensity": 1/2, "range": 1/2, "filtered": 0, "fused": 0.0}
valid_keys = [k for k in weights if k in dat["arrays"] and weights[k] > 0]
for k in valid_keys: mods_hess[k] = compute_hessians_per_scale(to_gray(dat["arrays"][k]), Σ)
fused_H = fuse_hessians_per_scale(mods_hess, weights)

# 2. Graph Construction
coords, _, S = build_frangi_similarity_graph(fused_H, β, c, c_θ, R, candidate_mask=None, threshold_mask=threshold_mask, dark_ridges=dark_ridges)
D = distances_from_similarity(S, mode="minus")
if K == 2: D = triangle_connectivity_graph(coords, D)
D_cc, idx_nodes = largest_connected_component(D)

print(f"Largest CC: {D_cc.shape[0]} nodes.")

# 3. HDBSCAN Clustering
sub_coords = coords[idx_nodes]
if D_cc.shape[0] > 0:
    # Dist = D_cc.copy()
    # Dist.data = Dist.data ** expZ
    # Dist = Dist.tocsr()
    # Dist.setdiag(0.0)

    # Explicit HDBSCAN call as requested
    # clusterer = hdbscan.HDBSCAN(
    #     metric="precomputed",
    #     min_cluster_size=min_cluster_size,
    #     min_samples=min_samples,
    #     max_dist=max_dist,
    #     allow_single_cluster=allow_single_cluster,
    # )
    # labels = clusterer.fit_predict(Dist)
    # --- MODIFICATION: SKIP HDBSCAN ---
    labels = np.zeros(D_cc.shape[0], dtype=int)
    # ----------------------------------

    # Detailed output
    print("Clusters:", np.unique(labels), ". 'Noise':", np.sum(labels == -1))
else:
    labels = np.array([])

# 4. Skeletonization (MST + Betweenness)
all_edges = []
sk_pred_mask = np.zeros_like(base, dtype=np.uint8)
centrality_vis = np.zeros_like(base, dtype=np.float32) # VISU

if labels.size > 0:
    for lab in np.unique(labels):
        if lab < 0: continue
        cl = np.where(labels == lab)[0]
        if cl.size < 3: continue
        mst = mst_on_cluster(D_cc, cl)
        global_indices = idx_nodes[cl]
        S_cluster = S[global_indices, :][:, global_indices]

        # --- VISU CENTRALITY START ---
        # Re-compute centrality map for visualization (same logic as in mst_kcenters)
        from scipy.sparse.csgraph import breadth_first_order
        N_cl = mst.shape[0]
        order, predecessors = breadth_first_order(mst, i_start=0, directed=False, return_predecessors=True)
        node_weights = np.ones(N_cl, dtype=np.float64)
        if S_cluster is not None:
             node_weights = S_cluster.max(axis=1).toarray().flatten().astype(np.float64)
        subtree_mass = node_weights.copy()
        for i in order[::-1]:
            if i != 0:
                p = predecessors[i]
                if p>=0 and p<N_cl: subtree_mass[p] += subtree_mass[i]
        total_mass = subtree_mass[0]
        cent = subtree_mass * (total_mass - subtree_mass)
        # Normalize for display
        if cent.max() > 0: cent /= cent.max()

        # Map back to image
        rows, cols = sub_coords[cl][:, 0], sub_coords[cl][:, 1]
        centrality_vis[rows, cols] = cent
        # --- VISU CENTRALITY END ---

        nodes_kept, skel_graph = extract_backbone_centrality(mst, f_threshold=f_threshold, S=S_cluster, take_similarity=True, min_centrality=min_centrality)
        segs = skeleton_from_mst_graph(skel_graph, sub_coords[cl], nodes_kept, S=S_cluster, take_similarity=True)
        if segs.shape[0] > 0: all_edges.append(segs)

    if all_edges:
        fault_edges = np.vstack(all_edges)
        for e in fault_edges:
            r0, c0, r1, c1, _ = e
            rr, cc = np.linspace(r0, r1, int(max(abs(r1-r0), abs(c1-c0))+1)), np.linspace(c0, c1, int(max(abs(r1-r0), abs(c1-c0))+1))
            rr, cc = np.clip(rr.astype(int), 0, sk_pred_mask.shape[0]-1), np.clip(cc.astype(int), 0, sk_pred_mask.shape[1]-1)
            sk_pred_mask[rr, cc] = 1

# 5. Visualizations
# Compute Best Scale features
l2_stack = np.stack([Hd['e2n'] for Hd in fused_H], axis=0)
e1_stack = np.stack([Hd['e1n'] for Hd in fused_H], axis=0)
theta_stack = np.stack([Hd['theta'] for Hd in fused_H], axis=0)

# Max over scales
best_idx = np.argmax(np.abs(l2_stack), axis=0)
H, W = l2_stack.shape[1], l2_stack.shape[2]
yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')

l2_fused_vis = l2_stack[best_idx, yy, xx]
e1_fused_vis = e1_stack[best_idx, yy, xx]
theta_fused_vis = theta_stack[best_idx, yy, xx]

ratio_vis = np.abs(e1_fused_vis) / (np.abs(l2_fused_vis) + 1e-12)
ratio_vis = np.clip(ratio_vis, 0, 1) # Frangi ratio is usually in [0,1] for tubes

sim_vis = np.zeros_like(base, dtype=np.float32)
# if S.shape[0] > 0:
degs = np.array(S.max(axis=1).toarray()).flatten()
sim_vis[coords[:,0], coords[:,1]] = degs

# Prepare Metrics Visualization (Thick Skeletons)
sk_pred_thick = thicken(sk_pred_mask, pixels=3)
sk_gt_thick = thicken(skeletonize_lee(gt), pixels=3)

fig = plt.figure(figsize=(20, 25))

# Row 1: Inputs
plt.subplot(5, 4, 1); plt.title("Intensity"); plt.imshow(dat["arrays"]["intensity"], cmap="gray"); plt.axis("off")
plt.subplot(5, 4, 2); plt.title("Range"); plt.imshow(dat["arrays"]["range"], cmap="gray"); plt.axis("off")

if 'fused' in dat['arrays']:
    plt.subplot(5, 4, 3); plt.title("Fused Input"); plt.imshow(dat["arrays"]["fused"], cmap="gray"); plt.axis("off")
else:
    plt.subplot(5, 4, 3); plt.title("Fused Input (N/A)"); plt.axis("off")

if 'filtered' in dat['arrays']:
    plt.subplot(5, 4, 4); plt.title("Filtered Range"); plt.imshow(dat["arrays"]["filtered"], cmap="gray"); plt.axis("off")
else:
    plt.subplot(5, 4, 4); plt.axis("off")

# Row 2: Hessian Geometry
plt.subplot(5, 4, 5); plt.title("Frangi Ratio (|λ1|/|λ2|)"); plt.imshow(ratio_vis, cmap="viridis"); plt.axis("off")
plt.subplot(5, 4, 6); plt.title("Hessian Angle sin(θ)"); plt.imshow(np.sin(theta_fused_vis), cmap="twilight"); plt.axis("off")
plt.subplot(5, 4, 7); plt.axis("off") # Empty
plt.subplot(5, 4, 8); plt.axis("off") # Empty

# Row 3: Intermediate Features
plt.subplot(5, 4, 9); plt.title("Fused |λ2|"); plt.imshow(np.abs(l2_fused_vis), cmap="magma"); plt.axis("off")
plt.subplot(5, 4, 10); plt.title("Frangi Similarity (Max)"); plt.imshow(sim_vis, cmap="inferno"); plt.axis("off")

ax_cl = plt.subplot(5, 4, 11); ax_cl.set_title("Connected Component (No Clustering)")
ax_cl.imshow(base, cmap='gray', alpha=0.5)
if labels.size > 0:
    noise_mask = labels == -1
    cluster_mask = labels >= 0
    if np.any(noise_mask):
        ax_cl.scatter(sub_coords[noise_mask, 1], sub_coords[noise_mask, 0], c='k', s=0.5, alpha=0.3, label='Noise')
    if np.any(cluster_mask):
        ax_cl.scatter(sub_coords[cluster_mask, 1], sub_coords[cluster_mask, 0], c=labels[cluster_mask], cmap='tab20', s=1)
ax_cl.axis('off')

ax_cent = plt.subplot(5, 4, 12); ax_cent.set_title("Betweenness Centrality")
im_cent = ax_cent.imshow(centrality_vis, cmap="inferno")
# plt.colorbar(im_cent, ax=ax_cent, fraction=0.046, pad=0.04)
ax_cent.axis('off')

# Row 4: Results
plt.subplot(5, 4, 13); plt.title("Ground Truth"); plt.imshow(gt, cmap="gray"); plt.axis("off")

csd_path = f"/content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff/20000_1000/test_output_fused/im{seed+1:05d}_output_ens.png"
if os.path.exists(csd_path):
    try:
        csd_img = np.array(Image.open(csd_path).convert('L'))
        csd_bin = (csd_img > 127).astype(np.uint8)
        csd_bin = binary_closing(csd_bin, footprint=disk(2))
        csd_bin = binary_opening(csd_bin, footprint=disk(2))
        plt.subplot(5, 4, 14); plt.title("CrackSegDiff"); plt.imshow(csd_bin, cmap="Greens"); plt.axis("off")
    except Exception as e: print("Error loading CSD:", e)
else:
    plt.subplot(5, 4, 14); plt.axis("off")

ax_sk = plt.subplot(5, 4, 15); ax_sk.set_title("Ours: Skeleton Overlay")
ax_sk.imshow(base, cmap='gray')
ax_sk.imshow(sk_pred_mask, cmap='Reds', alpha=0.3, vmin=0, vmax=1)
ax_sk.axis('off')

ax_ov = plt.subplot(5, 4, 16); ax_ov.set_title("Overlay: GT(W)+Ours(R)+CSD(G)")
ax_ov.imshow(np.zeros_like(base), cmap='gray')
ax_ov.imshow(gt, cmap='Greys', alpha=0.5)
ax_ov.imshow(sk_pred_mask, cmap='Reds', alpha=0.5)
if 'csd_bin' in locals():
    ax_ov.imshow(csd_bin, cmap='Greens', alpha=0.3)
ax_ov.axis('off')

# Row 5: Metrics Visualization (Thick Skeletons)
plt.subplot(5, 4, 17); plt.title("GT Thick (Target)"); plt.imshow(sk_gt_thick, cmap="Greys"); plt.axis("off")

if 'csd_bin' in locals():
    sk_csd_thick = thicken(skeletonize_lee(csd_bin), pixels=3)
    plt.subplot(5, 4, 18); plt.title("CSD Thick (Comp)"); plt.imshow(sk_csd_thick, cmap="Greens"); plt.axis("off")
else:
    plt.subplot(5, 4, 18); plt.axis("off")

plt.subplot(5, 4, 19); plt.title("Ours Thick (Pred)"); plt.imshow(sk_pred_thick, cmap="Reds"); plt.axis("off")

ax_ov_thick = plt.subplot(5, 4, 20); ax_ov_thick.set_title("Metric Overlay")
ax_ov_thick.imshow(np.zeros_like(base), cmap='gray')
ax_ov_thick.imshow(sk_gt_thick, cmap='Greys', alpha=0.5)
ax_ov_thick.imshow(sk_pred_thick, cmap='Reds', alpha=0.5)
if 'sk_csd_thick' in locals():
    ax_ov_thick.imshow(sk_csd_thick, cmap='Greens', alpha=0.3)
ax_ov_thick.axis('off')

plt.tight_layout()
plt.show()

# Metrics for this image
print("--- Metrics (Seed 1) ---")
print("Jaccard:", jaccard_index(sk_pred_thick, sk_gt_thick))
print("Tversky:", tversky_index(sk_pred_thick, sk_gt_thick, alpha=1.0, beta=0.5))
print("Wasserstein:", wasserstein_distance_skeletons(sk_pred_thick, sk_gt_thick))

In [ ]:
# --- Batch Processing 500 Images (USE_COMBO=True) ---
import itertools
import hashlib
import json

# # GT not relevant : 1, 39, 42, 152, 203, 204, 206, 397, 411, 414, 415, 431, 449, 452, 457, 460, 461, 465, 469, 471, 475, 478
# # Illustration : 203 ou 206
# # Question : 490 ?!?!
excluded_ids = [1, 39, 42, 133, 152, 203, 204, 206, 397, 411, 414, 415, 431, 449, 452, 457, 460, 461, 465, 469, 471, 475, 478]
excluded_ids = [idx -1 for idx in excluded_ids]

start_idx = 0
end_idx = 500
n_jobs = 8
USE_COMBO = False
COMPUTE_RESULTS = True # @param {type:"boolean"}

# Parameters dict for logging
params_log = {
    "sigma": Σ,
    "beta": β,
    "c": c,
    "c_theta": c_θ,
    "R": R,
    "K": K,
    "threshold_mask": threshold_mask,
    "f_threshold": f_threshold,
    "min_centrality": min_centrality,
    "weights": weights,
    "dark_ridges": dark_ridges
}

# Generate descriptive name
w_str = "-".join([f"{k[0]}{v:.2f}" for k,v in weights.items()])
run_name = f"Batch_beta{β}_R{R}_w{w_str}"

base_output_dir = "/content/drive/MyDrive/Datasets/FIND/Results/Avignon_Notebook_Batch"
output_dir = os.path.join(base_output_dir, run_name)
os.makedirs(output_dir, exist_ok=True)

# Save params
with open(os.path.join(output_dir, "params.json"), "w") as f:
    json.dump(params_log, f, indent=4)

print(f"Results directory: {output_dir}")

def process_image_idx_combo(idx):
    if idx in excluded_ids: return None
    try:
        dat = load_modalities_and_gt_by_index(struct, idx)
        base = dat["arrays"].get("intensity", next(iter(dat["arrays"].values())))
        gt = (dat["arrays"].get("label", np.zeros_like(base)) > 0).astype(np.uint8)
        gt = binary_closing(gt, footprint=disk(2))
        gt = binary_opening(gt, footprint=disk(2))
        sk_gt_thick = thicken(skeletonize_lee(gt), pixels=3)

        # --- Load CSD & Compute Metrics ---
        csd_path_in = f"/content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff/20000_1000/test_output_fused/im{idx+1:05d}_output_ens.png"
        sk_csd_thick = np.zeros_like(sk_gt_thick)
        csd_jac, csd_tvs, csd_wass = 0.0, 0.0, 0.0
        if os.path.exists(csd_path_in):
            try:
                csd_img = np.array(Image.open(csd_path_in).convert('L'))
                csd_bin = (csd_img > 127).astype(np.uint8)
                csd_bin = binary_closing(csd_bin, footprint=disk(2))
                csd_bin = binary_opening(csd_bin, footprint=disk(2))
                sk_csd_thick = thicken(skeletonize_lee(csd_bin), pixels=3)
                csd_jac = jaccard_index(sk_csd_thick, sk_gt_thick)
                csd_tvs = tversky_index(sk_csd_thick, sk_gt_thick, alpha=1.0, beta=0.5)
                csd_wass = wasserstein_distance_skeletons(sk_csd_thick, sk_gt_thick)
            except: pass

        # Pre-compute Hessians
        hessian_cache = {}
        valid_keys = [k for k in weights if k in dat["arrays"] and weights[k] > 0]
        for k in valid_keys:
            arr = to_gray(dat["arrays"][k])
            h_norm = compute_hessians_per_scale(arr, Σ)
            hessian_cache[k] = [h_norm]
            if USE_COMBO:
                h_inv = compute_hessians_per_scale(255 - arr, Σ)
                hessian_cache[k].append(h_inv)

        combo_indices = list(itertools.product([0, 1], repeat=len(valid_keys))) if USE_COMBO else [tuple(0 for _ in valid_keys)]
        best_tversky = -1.0
        best_res = None

        for combo in combo_indices:
            current_mods = {}
            for i, mod in enumerate(valid_keys):
                current_mods[mod] = hessian_cache[mod][combo[i]]
            fused_H = fuse_hessians_per_scale(current_mods, weights)
            coords, _, S = build_frangi_similarity_graph(fused_H, β, c, c_θ, R, candidate_mask=None, threshold_mask=threshold_mask, dark_ridges=dark_ridges)
            D = distances_from_similarity(S, mode="minus")
            if K == 2: D = triangle_connectivity_graph(coords, D)
            D_cc, idx_nodes = largest_connected_component(D)
            sk_pred = np.zeros_like(base, dtype=np.uint8)
            if D_cc.shape[0] > 0:
                labels = np.zeros(D_cc.shape[0], dtype=int) # SKIP HDBSCAN
                sub_coords = coords[idx_nodes]
                all_edges = []
                for lab in np.unique(labels):
                    if lab < 0: continue
                    cl = np.where(labels == lab)[0]
                    if cl.size < 3: continue
                    mst = mst_on_cluster(D_cc, cl)
                    global_indices = idx_nodes[cl]
                    S_cluster = S[global_indices, :][:, global_indices]
                    nodes_kept, skel_graph = extract_backbone_centrality(mst, f_threshold=f_threshold, S=S_cluster, take_similarity=True, min_centrality=min_centrality)
                    segs = skeleton_from_mst_graph(skel_graph, sub_coords[cl], nodes_kept, S=S_cluster, take_similarity=True)
                    if segs.shape[0] > 0: all_edges.append(segs)
                if all_edges:
                    fault_edges = np.vstack(all_edges)
                    mask = np.zeros_like(base, dtype=np.uint8)
                    for e in fault_edges:
                        r0, c0, r1, c1, _ = e
                        rr, cc = np.linspace(r0, r1, int(max(abs(r1-r0), abs(c1-c0))+1)), np.linspace(c0, c1, int(max(abs(r1-r0), abs(c1-c0))+1))
                        rr, cc = np.clip(rr.astype(int), 0, mask.shape[0]-1), np.clip(cc.astype(int), 0, mask.shape[1]-1)
                        mask[rr, cc] = 1
                    sk_pred = skeletonize_lee(mask)
            sk_pred_thick = thicken(sk_pred, pixels=3)
            tvs = tversky_index(sk_pred_thick, sk_gt_thick, alpha=1.0, beta=0.5)
            if tvs > best_tversky:
                best_tversky = tvs
                jac = jaccard_index(sk_pred_thick, sk_gt_thick)
                wass = wasserstein_distance_skeletons(sk_pred_thick, sk_gt_thick)
                # --- SAVE VISUALIZATIONS ---
                skel_dir = os.path.join(output_dir, "skeleton")
                over_dir = os.path.join(output_dir, "overlay")
                os.makedirs(skel_dir, exist_ok=True)
                os.makedirs(over_dir, exist_ok=True)
                iio.imwrite(os.path.join(skel_dir, f"im{idx+1:05d}_skel.png"), (sk_pred_thick * 255).astype(np.uint8))
                # RGB Overlay
                H_ov, W_ov = sk_gt_thick.shape
                ov_img = np.zeros((H_ov, W_ov, 3), dtype=np.uint8)
                ov_img[..., 0] = np.clip(sk_gt_thick * 255 + sk_pred_thick * 255, 0, 255)
                ov_img[..., 1] = np.clip(sk_gt_thick * 255 + sk_csd_thick * 255, 0, 255)
                ov_img[..., 2] = np.clip(sk_gt_thick * 255, 0, 255)
                iio.imwrite(os.path.join(over_dir, f"im{idx+1:05d}_overlay.png"), ov_img)
                best_res = {
                    "Image": f"im{idx+1:05d}",
                    "Jaccard": jac, "Tversky": tvs, "Wasserstein": wass,
                    "CSD_Jaccard": csd_jac, "CSD_Tversky": csd_tvs, "CSD_Wasserstein": csd_wass,
                    "Combo": str(combo)
                }
        return best_res
    except Exception as e: return None

if COMPUTE_RESULTS:
    print(f"Processing batch {start_idx}-{end_idx}...")
    with tqdm_joblib(tqdm(total=end_idx-start_idx)) as progress_bar:
        results = Parallel(n_jobs=n_jobs)(delayed(process_image_idx_combo)(i) for i in range(start_idx, end_idx))
    results = [r for r in results if r is not None]
    df_res = pd.DataFrame(results)
    if not df_res.empty:
        print("\n--- Results ---")
        mean_vals = df_res[["Jaccard", "Tversky", "Wasserstein", "CSD_Jaccard", "CSD_Tversky", "CSD_Wasserstein"]].mean()
        print("--- Mean Results ---")
        print(mean_vals)
        mean_row = mean_vals.to_dict()
        mean_row["Image"] = "MEAN"
        mean_row["Combo"] = "N/A"
        df_final = pd.concat([df_res, pd.DataFrame([mean_row])], ignore_index=True)
        df_final.to_csv(os.path.join(output_dir, "metrics_combo.csv"), index=False)
    else: print("No valid results.")
else:
    print("COMPUTE_RESULTS is False. Skipping batch computation.")
    csv_path = os.path.join(output_dir, "metrics_combo.csv")
    if os.path.exists(csv_path):
        print(f"Loading results from {csv_path}")
        df_final = pd.read_csv(csv_path)
        print("--- Loaded Mean Results ---")
        print(df_final.tail(1))
    else:
        print(f"No existing results found at {csv_path}")


# Robustness Analysis: Noisy Images

In [ ]:
import os
import numpy as np
from PIL import Image
from skimage.morphology import binary_closing, binary_opening, disk

# # Clean: exactly the pattern used
CSD_CLEAN_DIR = "/content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff/20000_1000/test_output_fused"

# # Noisy: adapt to your storage. Proposed convention:
#   /content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff_noise/<exp>/<tag>/test_output_fused/imXXXXX_output_ens.png
CSD_NOISE_ROOT = "/content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff_noise"

def _noise_tag(x: float, ndigits: int = 4):
    return f"{x:.{ndigits}f}".replace(".", "p")

def csd_pred_path(idx: int, exp_name: str, level: float):
    idx1 = idx + 1
    if exp_name == "clean":
        return os.path.join(CSD_CLEAN_DIR, f"im{idx1:05d}_output_ens.png")
    tag = _noise_tag(level)
    return os.path.join(CSD_NOISE_ROOT, exp_name, tag, "test_output_fused", f"im{idx1:05d}_output_ens.png")

def load_csd_thick(idx: int, exp_name: str, level: float):
    """
    Returns the thick CrackSegDiff skeleton (like your code: threshold->closing/opening->skeletonize->thicken),
    or None if the file does not exist.
    """
    path = csd_pred_path(idx, exp_name, level)
    if not os.path.exists(path):
        return None

    try:
        csd_img = np.array(Image.open(path).convert("L"))
        csd_bin = (csd_img > 127).astype(np.uint8)
        csd_bin = binary_closing(csd_bin, footprint=disk(2))
        csd_bin = binary_opening(csd_bin, footprint=disk(2))
        sk_csd_thick = thicken(skeletonize_lee(csd_bin), pixels=THICK_PIXELS)
        return sk_csd_thick
    except Exception:
        return None


In [ ]:
# --- *** Generate and Save Noisy Datasets (Optional) *** ---
import os
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as iio
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

save_noisy_images_on_drive = False # @param {type:"boolean"}
NOISE_SAVE_ROOT = "/content/drive/MyDrive/Datasets/FIND/Noisy"

# --- Re-definition of Noise Functions for Standalone Execution ---
# (Must match the Benchmark logic exactly to ensure seeds are identical)
NOISE_BASE_SEED = 1

def _normalize01(x: np.ndarray):
    x = np.asarray(x).astype(np.float32)
    mn = float(np.min(x))
    mx = float(np.max(x))
    if (mx - mn) < 1e-12:
        return np.zeros_like(x, dtype=np.float32), mn, mx
    return (x - mn) / (mx - mn), mn, mx

def _denormalize01(x01: np.ndarray, mn: float, mx: float):
    return x01 * (mx - mn) + mn

def add_speckle_intensity(x: np.ndarray, var: float, rng: np.random.Generator):
    if var <= 0: return np.asarray(x).astype(np.float32)
    x01, mn, mx = _normalize01(x)
    n = rng.normal(0.0, np.sqrt(var), size=x01.shape).astype(np.float32)
    y01 = x01 + x01 * n
    y01 = np.clip(y01, 0.0, 1.0)
    return _denormalize01(y01, mn, mx).astype(np.float32)

def add_gaussian_range(x: np.ndarray, sigma: float, rng: np.random.Generator):
    if sigma <= 0: return np.asarray(x).astype(np.float32)
    x01, mn, mx = _normalize01(x)
    y01 = x01 + rng.normal(0.0, sigma, size=x01.shape).astype(np.float32)
    y01 = np.clip(y01, 0.0, 1.0)
    return _denormalize01(y01, mn, mx).astype(np.float32)

def make_noisy_arrays(arrays: dict, idx: int, level_id: int, speckle_var: float = 0.0, range_sigma: float = 0.0, noise_filtered_like_range: bool = True):
    out = dict(arrays)
    # Critical: Seed logic must match the benchmark exactly
    rng_I = np.random.default_rng(NOISE_BASE_SEED + 100000 * idx + 97 * level_id + 1)
    rng_R = np.random.default_rng(NOISE_BASE_SEED + 100000 * idx + 97 * level_id + 2)
    if "intensity" in out and speckle_var > 0:
        out["intensity"] = add_speckle_intensity(out["intensity"], speckle_var, rng_I)
    if "range" in out and range_sigma > 0:
        out["range"] = add_gaussian_range(out["range"], range_sigma, rng_R)
    if noise_filtered_like_range and ("filtered" in out) and range_sigma > 0:
        out["filtered"] = add_gaussian_range(out["filtered"], range_sigma, rng_R)
    return out

def save_single_image_noisy(idx, struct, exp_name, level_id, lvl, speckle_var, range_sigma):
    try:
        # Load Data
        dat = load_modalities_and_gt_by_index(struct, idx)

        # Generate Noisy
        noisy_arrays = make_noisy_arrays(dat["arrays"], idx, level_id, speckle_var, range_sigma, noise_filtered_like_range=True)

        # Format Path: Root / exp / tag / imXXXXX_modality.png
        tag = f"{lvl:.4f}".replace(".", "p")
        out_dir = os.path.join(NOISE_SAVE_ROOT, exp_name, tag)
        os.makedirs(out_dir, exist_ok=True) # Race condition handled by OS usually fine, or pre-create

        base_name = f"im{idx+1:05d}"

        # 1. Save Intensity (Grayscale)
        if "intensity" in noisy_arrays:
            img = noisy_arrays["intensity"]
            # Normalize for visualization 0-255
            img_norm = (img - img.min()) / (img.max() - img.min() + 1e-8)
            img_uint8 = (img_norm * 255).astype(np.uint8)

            fname = f"{base_name}_intensity.png"
            iio.imwrite(os.path.join(out_dir, fname), img_uint8)

        # 2. Save Range (Jet Colormap)
        if "range" in noisy_arrays:
            rng_img = noisy_arrays["range"]
            # Normalize strict 0-1 for colormap
            rng_norm = (rng_img - rng_img.min()) / (rng_img.max() - rng_img.min() + 1e-8)

            # Apply Jet
            cmap = plt.get_cmap('jet')
            rgba_img = cmap(rng_norm) # Returns (H, W, 4) floats
            rgb_img = rgba_img[:, :, :3] # Keep RGB

            rgb_uint8 = (rgb_img * 255).astype(np.uint8)

            fname = f"{base_name}_range.png"
            iio.imwrite(os.path.join(out_dir, fname), rgb_uint8)

    except Exception as e:
        print(f"Error saving idx {idx}: {e}")

# --- Execution Block ---
if save_noisy_images_on_drive:
    print(f"Generating and saving noisy images to {NOISE_SAVE_ROOT}...")

    # Configuration (Must match benchmark)
    speckle_vars = [0.0, 0.01, 0.05, 0.10, 0.3, 0.5]
    range_sigmas = [0.0, 0.01, 0.05, 0.10, 0.3, 0.5]
    experiments = [
        ("speckle_intensity", speckle_vars),
        ("gauss_range", range_sigmas),
        ("both", range_sigmas)
    ]

    excluded_ids = [1, 39, 42, 133, 152, 203, 204, 206, 397, 411, 414, 415, 431, 449, 452, 457, 460, 461, 465, 469, 471, 475, 478]
    excluded_ids = [i-1 for i in excluded_ids]
    indices = [i for i in range(500) if i not in excluded_ids]

    for exp_name, levels in experiments:
        print(f"Processing experiment: {exp_name}")
        for level_id, lvl in enumerate(levels):
            # Determine params
            if exp_name == "speckle_intensity": sp, sg = lvl, 0.0
            elif exp_name == "gauss_range": sp, sg = 0.0, lvl
            elif exp_name == "both": sp, sg = lvl, lvl

            # Run Parallel Saving
            with tqdm_joblib(tqdm(total=len(indices), desc=f"Saving {exp_name} {lvl}")):
                Parallel(n_jobs=8)(delayed(save_single_image_noisy)(
                    idx, struct, exp_name, level_id, lvl, sp, sg
                ) for idx in indices)

    print("Done saving images.")
else:
    print("Skipping image generation (checkbox unchecked).")

In [ ]:
# --- Visualize Noisy Example (Seed 1) ---
# We reuse the logic but specifically for a noisy instance
# Let's pick an interesting noise level to visualize, e.g., Speckle=0.1 or Range=0.1
# Or just generate one on the fly.

VIZ_IDX = 1 # Same seed as before
VIZ_SPECKLE = 0.5
VIZ_RANGE_SIGMA = 0.5

print(f"Visualizing Noisy Example (Idx {VIZ_IDX}, Speckle {VIZ_SPECKLE}, RangeSigma {VIZ_RANGE_SIGMA})... FIXED")

# 1. Generate Noisy Arrays
dat = load_modalities_and_gt_by_index(struct, VIZ_IDX)
# Use the helper function defined in the Save cell (ensure it's run or re-define if needed)
# We re-define minimal versions here to be safe if that cell wasn't run
def _norm01(x):
    x = np.asarray(x).astype(np.float32)
    mn, mx = x.min(), x.max()
    return (x - mn)/(mx - mn + 1e-12) if (mx-mn)>1e-12 else np.zeros_like(x), mn, mx
def _denorm01(x, mn, mx): return x*(mx-mn) + mn

rng_I = np.random.default_rng(NOISE_BASE_SEED + 100000 * VIZ_IDX + 97 * 0 + 1) # Dummy level_id 0
rng_R = np.random.default_rng(NOISE_BASE_SEED + 100000 * VIZ_IDX + 97 * 0 + 2)

# Intensity Noise
i01, mn, mx = _norm01(dat["arrays"]["intensity"])
n = rng_I.normal(0.0, np.sqrt(VIZ_SPECKLE), size=i01.shape).astype(np.float32)
i_noisy = _denorm01(np.clip(i01 + i01*n, 0, 1), mn, mx)

# Range Noise
r01, mn, mx = _norm01(dat["arrays"]["range"])
r_noisy = _denorm01(np.clip(r01 + rng_R.normal(0.0, VIZ_RANGE_SIGMA, size=r01.shape), 0, 1), mn, mx)

# 2. Compute Hessians & Fuse
hess_i = compute_hessians_per_scale(to_gray(i_noisy), Σ)
hess_r = compute_hessians_per_scale(to_gray(r_noisy), Σ)
mods_noisy = {"intensity": hess_i, "range": hess_r}
weights_noisy = {"intensity": 0.5, "range": 0.5}
fused_H_noisy = fuse_hessians_per_scale(mods_noisy, weights_noisy)

# 3. Graph & Extraction
coords, _, S = build_frangi_similarity_graph(fused_H_noisy, β, c, c_θ, R, candidate_mask=None, threshold_mask=threshold_mask, dark_ridges=dark_ridges)
D = distances_from_similarity(S, mode="minus")
if K == 2: D = triangle_connectivity_graph(coords, D)
D_cc, idx_nodes = largest_connected_component(D)

mask_noisy = np.zeros_like(i_noisy)
centrality_noisy = np.zeros_like(i_noisy)

if D_cc.shape[0] > 0:
    labels = np.zeros(D_cc.shape[0], dtype=int)
    sub_coords = coords[idx_nodes]
    all_edges = []
    for lab in np.unique(labels):
        if lab < 0: continue
        cl = np.where(labels == lab)[0]
        if cl.size < 3: continue
        mst = mst_on_cluster(D_cc, cl)
        global_indices = idx_nodes[cl]
        S_cluster = S[global_indices, :][:, global_indices]

        # Centrality Vis
        from scipy.sparse.csgraph import breadth_first_order
        N_cl = mst.shape[0]
        order, preds = breadth_first_order(mst, i_start=0, directed=False, return_predecessors=True)
        nw = S_cluster.max(axis=1).toarray().flatten() if S_cluster is not None else np.ones(N_cl)
        mass = nw.copy()
        for i in order[::-1]:
            if i!=0:
                p = preds[i]
                if p>=0: mass[p]+=mass[i]
        cent = mass * (mass[0] - mass)
        if cent.max()>0: cent/=cent.max()
        r, c_ = sub_coords[cl,0], sub_coords[cl,1]
        centrality_noisy[r, c_] = cent

        nodes_kept, skel_graph = extract_backbone_centrality(mst, f_threshold=f_threshold, S=S_cluster, take_similarity=True, min_centrality=min_centrality)
        segs = skeleton_from_mst_graph(skel_graph, sub_coords[cl], nodes_kept, S=S_cluster, take_similarity=True)
        if segs.shape[0] > 0: all_edges.append(segs)

    if all_edges:
        fault_edges = np.vstack(all_edges)
        for e in fault_edges:
            r0, c0, r1, c1, _ = e
            rr, cc = np.linspace(r0, r1, int(max(abs(r1-r0), abs(c1-c0))+1)), np.linspace(c0, c1, int(max(abs(r1-r0), abs(c1-c0))+1))
            rr, cc = np.clip(rr.astype(int), 0, mask_noisy.shape[0]-1), np.clip(cc.astype(int), 0, mask_noisy.shape[1]-1)
            mask_noisy[rr, cc] = 1.0

# 4. Features
l2_stack = np.stack([Hd['e2n'] for Hd in fused_H_noisy], axis=0)
best_idx = np.abs(l2_stack).argmax(axis=0)
H, W = l2_stack.shape[1], l2_stack.shape[2]
yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
l2_vis = l2_stack[best_idx, yy, xx]

sim_vis = np.zeros_like(i_noisy)
if S.shape[0] > 0:
    degs = np.array(S.max(axis=1).toarray()).flatten()
    sim_vis[coords[:,0], coords[:,1]] = degs

# 5. Plot
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
# Row 1: Inputs
axes[0,0].imshow(i_noisy, cmap='gray'); axes[0,0].set_title(f"Noisy Intensity (Var={VIZ_SPECKLE})"); axes[0,0].axis('off')
axes[0,1].imshow(r_noisy, cmap='gray'); axes[0,1].set_title(f"Noisy Range (Sig={VIZ_RANGE_SIGMA})"); axes[0,1].axis('off')
axes[0,2].imshow(np.abs(l2_vis), cmap='magma'); axes[0,2].set_title("Fused |λ2|"); axes[0,2].axis('off')
axes[0,3].imshow(sim_vis, cmap='inferno'); axes[0,3].set_title("Similarity"); axes[0,3].axis('off')

# Row 2: Graph & Result
axes[1,0].imshow(centrality_noisy, cmap='inferno'); axes[1,0].set_title("Centrality"); axes[1,0].axis('off')
axes[1,1].imshow(dat['arrays']['label'], cmap='gray'); axes[1,1].set_title("GT"); axes[1,1].axis('off')
axes[1,2].imshow(i_noisy, cmap='gray')
axes[1,2].imshow(mask_noisy, cmap='Reds', alpha=0.5); axes[1,2].set_title("Result Overlay"); axes[1,2].axis('off')
axes[1,3].axis('off')

plt.suptitle(f"Noisy Example Visualization (Seed {VIZ_IDX})")
plt.tight_layout()
plt.show()

In [ ]:
# --- Consolidated and Robust Noise Benchmark ---
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
import itertools
from PIL import Image
import imageio.v2 as iio
import traceback
from skimage.morphology import binary_closing, binary_opening, disk

# --- 1. CSD Path Helpers ---
CSD_NOISE_ROOT = "/content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff_noise"

def _noise_tag(x: float, ndigits: int = 4):
    return f"{x:.{ndigits}f}".replace(".", "p")

def get_csd_noisy_path(idx, exp_name, level):
    tag = _noise_tag(level)
    base_name = f"im{idx+1:05d}"
    # Structure: Root / Exp / Tag / test_output_fused / [file]
    folder = os.path.join(CSD_NOISE_ROOT, exp_name, tag, "test_output_fused")

    # Priority 1: imXXXXX.png (as per your description)
    p1 = os.path.join(folder, f"{base_name}.png")
    if os.path.exists(p1): return p1

    # Priority 2: imXXXXX_output_ens.png (standard CSD output)
    p2 = os.path.join(folder, f"{base_name}_output_ens.png")
    if os.path.exists(p2): return p2

    return p1 # Default to p1

# --- 2. Noise Functions ---
NOISE_BASE_SEED = 1

def _normalize01(x: np.ndarray):
    x = np.asarray(x).astype(np.float32)
    mn = float(np.min(x))
    mx = float(np.max(x))
    if (mx - mn) < 1e-12:
        return np.zeros_like(x, dtype=np.float32), mn, mx
    return (x - mn) / (mx - mn), mn, mx

def _denormalize01(x01: np.ndarray, mn: float, mx: float):
    return x01 * (mx - mn) + mn

def add_speckle_intensity(x: np.ndarray, var: float, rng: np.random.Generator):
    if var <= 0: return np.asarray(x).astype(np.float32)
    x01, mn, mx = _normalize01(x)
    n = rng.normal(0.0, np.sqrt(var), size=x01.shape).astype(np.float32)
    y01 = x01 + x01 * n
    y01 = np.clip(y01, 0.0, 1.0)
    return _denormalize01(y01, mn, mx).astype(np.float32)

def add_gaussian_range(x: np.ndarray, sigma: float, rng: np.random.Generator):
    if sigma <= 0: return np.asarray(x).astype(np.float32)
    x01, mn, mx = _normalize01(x)
    y01 = x01 + rng.normal(0.0, sigma, size=x01.shape).astype(np.float32)
    y01 = np.clip(y01, 0.0, 1.0)
    return _denormalize01(y01, mn, mx).astype(np.float32)

def make_noisy_arrays(arrays: dict, idx: int, level_id: int, speckle_var: float = 0.0, range_sigma: float = 0.0, noise_filtered_like_range: bool = True):
    out = dict(arrays)
    rng_I = np.random.default_rng(NOISE_BASE_SEED + 100000 * idx + 97 * level_id + 1)
    rng_R = np.random.default_rng(NOISE_BASE_SEED + 100000 * idx + 97 * level_id + 2)
    if "intensity" in out and speckle_var > 0:
        out["intensity"] = add_speckle_intensity(out["intensity"], speckle_var, rng_I)
    if "range" in out and range_sigma > 0:
        out["range"] = add_gaussian_range(out["range"], range_sigma, rng_R)
    if noise_filtered_like_range and ("filtered" in out) and range_sigma > 0:
        out["filtered"] = add_gaussian_range(out["filtered"], range_sigma, rng_R)
    return out

# --- 3. Frangi Prediction ---
def frangi_predict_mask_from_arrays(arrays: dict, weights: dict, use_combo: bool = False, combo: tuple = None, gt_thick: np.ndarray = None):
    # Ensure imports inside function for worker context safety
    from frangi_fusion import compute_hessians_per_scale, fuse_hessians_per_scale, build_frangi_similarity_graph, distances_from_similarity, triangle_connectivity_graph, largest_connected_component, mst_on_cluster, extract_backbone_centrality, skeleton_from_mst_graph, skeletonize_lee, thicken, to_gray, tversky_index
    import numpy as np
    import itertools

    base = arrays.get("intensity", next(iter(arrays.values())))
    base = np.asarray(base)
    valid_keys = [k for k in weights if (k in arrays and weights[k] > 0)]
    if len(valid_keys) == 0: raise ValueError("No valid modality found")

    hessian_cache = {}
    for k in valid_keys:
        arr = to_gray(arrays[k])
        h_norm = compute_hessians_per_scale(arr, Σ)
        hessian_cache[k] = [h_norm]
        if use_combo:
            h_inv = compute_hessians_per_scale(255 - arr, Σ)
            hessian_cache[k].append(h_inv)

    combo_list = [combo] if (combo is not None) else list(itertools.product([0, 1], repeat=len(valid_keys))) if use_combo else [tuple(0 for _ in valid_keys)]

    best_tversky = -1.0
    best_combo = combo_list[0]
    best_mask = np.zeros_like(base, dtype=np.uint8)

    for cmb in combo_list:
        current_mods = {}
        for i, mod in enumerate(valid_keys):
            current_mods[mod] = hessian_cache[mod][cmb[i]]
        fused_H = fuse_hessians_per_scale(current_mods, weights)
        coords, _, S = build_frangi_similarity_graph(fused_H, β, c, c_θ, R, candidate_mask=None, threshold_mask=threshold_mask, dark_ridges=dark_ridges)
        D = distances_from_similarity(S, mode="minus")
        if K == 2: D = triangle_connectivity_graph(coords, D)
        D_cc, idx_nodes = largest_connected_component(D)

        sk_pred_mask = np.zeros_like(base, dtype=np.uint8)
        if D_cc.shape[0] > 0:
            labels = np.zeros(D_cc.shape[0], dtype=int) # SKIP HDBSCAN
            sub_coords = coords[idx_nodes]
            all_edges = []
            for lab in np.unique(labels):
                if lab < 0: continue
                cl = np.where(labels == lab)[0]
                if cl.size < 3: continue
                mst = mst_on_cluster(D_cc, cl)
                global_indices = idx_nodes[cl]
                S_cluster = S[global_indices, :][:, global_indices]
                # Use globals f_threshold and min_centrality
                nodes_kept, skel_graph = extract_backbone_centrality(mst, f_threshold=f_threshold, S=S_cluster, take_similarity=True, min_centrality=min_centrality)
                segs = skeleton_from_mst_graph(skel_graph, sub_coords[cl], nodes_kept, S=S_cluster, take_similarity=True)
                if hasattr(segs, "shape") and segs.shape[0] > 0: all_edges.append(segs)
            if all_edges:
                fault_edges = np.vstack(all_edges)
                for e in fault_edges:
                    r0, c0, r1, c1, _ = e
                    n = int(max(abs(r1-r0), abs(c1-c0))+1)
                    rr, cc = np.linspace(r0, r1, n), np.linspace(c0, c1, n)
                    rr = np.clip(rr.astype(int), 0, sk_pred_mask.shape[0]-1)
                    cc = np.clip(cc.astype(int), 0, sk_pred_mask.shape[1]-1)
                    sk_pred_mask[rr, cc] = 1

        if use_combo and (combo is None) and (gt_thick is not None):
            sk_pred_thick = thicken(sk_pred_mask, pixels=3)
            tv = float(tversky_index(sk_pred_thick, gt_thick, alpha=1.0, beta=0.5))
            if tv > best_tversky:
                best_tversky = tv
                best_combo = cmb
                best_mask = sk_pred_mask
        else:
            best_mask = sk_pred_mask
            best_combo = cmb
            break
    return best_mask, best_combo

# --- 4. Process Function (Wrapper) ---
def process_image_noise(idx, struct, noise_exp_name, noise_levels, use_combo, weights, noise_filtered_like_range, combo_strategy, excluded_ids):
    if idx in excluded_ids: return None
    try:
        # Ensure imports for safety in parallel execution
        from frangi_fusion import jaccard_index, tversky_index, wasserstein_distance_skeletons, skeletonize_lee, thicken

        # Load data
        dat = load_modalities_and_gt_by_index(struct, idx)
        base = dat["arrays"].get("intensity", next(iter(dat["arrays"].values())))
        gt = (dat["arrays"].get("label", np.zeros_like(base)) > 0).astype(np.uint8)
        gt = binary_closing(gt, footprint=disk(2))
        gt = binary_opening(gt, footprint=disk(2))
        sk_gt_thick = thicken(skeletonize_lee(gt), pixels=3)

        fixed_combo = None
        if use_combo and combo_strategy == "freeze_clean":
            clean_mask, clean_combo = frangi_predict_mask_from_arrays(dat["arrays"], weights, use_combo=True, combo=None, gt_thick=sk_gt_thick)
            fixed_combo = clean_combo

        rows = []
        for level_id, lvl in enumerate(noise_levels):
            lvl = float(lvl)
            if noise_exp_name == "speckle_intensity": speckle_var, range_sigma = lvl, 0.0
            elif noise_exp_name == "gauss_range": speckle_var, range_sigma = 0.0, lvl
            elif noise_exp_name == "both": speckle_var, range_sigma = lvl, lvl
            else: raise ValueError(f"Unknown exp {noise_exp_name}")

            noisy_arrays = make_noisy_arrays(dat["arrays"], idx, level_id, speckle_var, range_sigma, noise_filtered_like_range)

            # --- A. Frangi Predict ---
            if use_combo and combo_strategy == "best_each":
                pred_mask, combo_used = frangi_predict_mask_from_arrays(noisy_arrays, weights, use_combo=True, combo=None, gt_thick=sk_gt_thick)
            else:
                combo_to_use = fixed_combo if use_combo else None
                pred_mask, combo_used = frangi_predict_mask_from_arrays(noisy_arrays, weights, use_combo=use_combo, combo=combo_to_use, gt_thick=None)

            # Metrics Ours
            sk_pred_thick = thicken(pred_mask, pixels=3)
            jac = float(jaccard_index(sk_pred_thick, sk_gt_thick))
            tvs = float(tversky_index(sk_pred_thick, sk_gt_thick, alpha=1.0, beta=0.5))
            wass = float(wasserstein_distance_skeletons(sk_pred_thick, sk_gt_thick))

            # --- B. CrackSegDiff (CSD) Metrics ---
            csd_jac, csd_tvs, csd_wass = np.nan, np.nan, np.nan
            try:
                csd_path = get_csd_noisy_path(idx, noise_exp_name, lvl)
                if os.path.exists(csd_path):
                    csd_img = np.array(Image.open(csd_path).convert('L'))
                    csd_bin = (csd_img > 127).astype(np.uint8)
                    csd_bin = binary_closing(csd_bin, footprint=disk(2))
                    csd_bin = binary_opening(csd_bin, footprint=disk(2))
                    sk_csd_thick = thicken(skeletonize_lee(csd_bin), pixels=3)

                    csd_jac = float(jaccard_index(sk_csd_thick, sk_gt_thick))
                    csd_tvs = float(tversky_index(sk_csd_thick, sk_gt_thick, alpha=1.0, beta=0.5))
                    csd_wass = float(wasserstein_distance_skeletons(sk_csd_thick, sk_gt_thick))
            except Exception:
                pass # Keep NaNs if file missing or error

            rows.append({
                "Image": idx+1, "NoiseExp": noise_exp_name, "NoiseLevel": lvl,
                "Jaccard": jac, "Tversky": tvs, "Wasserstein": wass,
                "CSD_Jaccard": csd_jac, "CSD_Tversky": csd_tvs, "CSD_Wasserstein": csd_wass
            })
        return rows
    except Exception as e:
        # LOG ERROR TO FILE
        with open("error_log.txt", "a") as f:
            f.write(f"Error idx {idx}: {str(e)}\n")
            f.write(traceback.format_exc() + "\n")
        return None

def run_noise_benchmark(struct, noise_exp_name, noise_levels, n_jobs, use_combo, weights, combo_strategy, noise_filtered_like_range, start_idx, end_idx, excluded_ids):
    indices = [i for i in range(start_idx, end_idx) if i not in excluded_ids]
    with tqdm_joblib(tqdm(total=len(indices), desc=f"Noise bench: {noise_exp_name}")):
        out = Parallel(n_jobs=n_jobs)(delayed(process_image_noise)(
            idx, struct, noise_exp_name, noise_levels, use_combo, weights, noise_filtered_like_range, combo_strategy, excluded_ids
        ) for idx in indices)
    rows = []
    for r in out:
        if r: rows.extend(r)
    return pd.DataFrame(rows)

# --- 5. Execution ---
weights = {"intensity": 1/2, "range": 1/2, "filtered": 0, "fused": 0.0}
speckle_vars = [0.0, 0.01, 0.05, 0.10, 0.3, 0.5]
range_sigmas = [0.0, 0.01, 0.05, 0.10, 0.3, 0.5]
USE_COMBO = False

# print("Running Speckle Benchmark...")
# df_speckle = run_noise_benchmark(struct, "speckle_intensity", speckle_vars, n_jobs=8, use_combo=USE_COMBO, weights=weights, combo_strategy="best_each", noise_filtered_like_range=True, start_idx=0, end_idx=500, excluded_ids=excluded_ids)
# print("Running Range Benchmark...")
# df_range = run_noise_benchmark(struct, "gauss_range", range_sigmas, n_jobs=8, use_combo=USE_COMBO, weights=weights, combo_strategy="best_each", noise_filtered_like_range=True, start_idx=0, end_idx=500, excluded_ids=excluded_ids)
print("Running Both Benchmark...")
df_both = run_noise_benchmark(struct, "both", range_sigmas, n_jobs=8, use_combo=USE_COMBO, weights=weights, combo_strategy="best_each", noise_filtered_like_range=True, start_idx=0, end_idx=500, excluded_ids=excluded_ids)

df_noise = pd.concat([df_both], ignore_index=True)
if not df_noise.empty:
    df_noise.to_csv("/content/drive/MyDrive/Datasets/FIND/Results/noise_robustness_metrics.csv", index=False)
    print("Saved results.")
else:
    print("No results! Check error_log.txt")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import matplotlib.ticker as ticker

# # Plotting style configuration
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

def display_noise_results(df):
    if df is None or df.empty:
        print("The DataFrame is empty or undefined.")
        return

    # 1. TABLEAU SYNTHÉTIQUE
    display(Markdown("### 📊 Numerical Synthesis (Mean ± Std Dev)"))
    metrics_map = {
        "Jaccard": "Jaccard",
        "Tversky": "Tversky",
        "Wasserstein": "Wasserstein",
        "CSD_Jaccard": "CSD Jac.",
        "CSD_Tversky": "CSD Tvs.",
        "CSD_Wasserstein": "CSD Wass."
    }
    avail_cols = [c for c in metrics_map.keys() if c in df.columns]
    grouped = df.groupby(['NoiseExp', 'NoiseLevel'])[avail_cols].agg(['mean', 'std'])
    summary_df = pd.DataFrame(index=grouped.index)
    for col in avail_cols:
        short_name = metrics_map[col]
        mean_col = grouped[col]['mean']
        std_col = grouped[col]['std']
        summary_df[short_name] = mean_col.apply(lambda x: f"{x:.3f}" if pd.notnull(x) else "-") + \
                                 " ± " + \
                                 std_col.apply(lambda x: f"{x:.3f}" if pd.notnull(x) else "-")
    display(summary_df.style.set_properties(**{'text-align': 'center'}).set_table_styles([
        dict(selector='th', props=[('text-align', 'center')])
    ]))

    # 2. VISUALISATION GRAPHIQUE
    display(Markdown("### 📈 Robustness Curves (Individual Figures)"))

    experiments = df['NoiseExp'].unique()

    # Configuration des métriques
    # ColName, CSDColName, YLim, YStep
    metrics_config = [
        ("Jaccard", "CSD_Jaccard", (0, 1), 0.1),
        ("Tversky", "CSD_Tversky", (0, 1), 0.1),
        ("Wasserstein", "CSD_Wasserstein", (0, 50), 5)
    ]

    for exp in experiments:
        subset = df[df['NoiseExp'] == exp]
        if subset.empty: continue

        display(Markdown(f"#### Experiment: {exp}"))

        for (our_col, csd_col, ylims, ystep) in metrics_config:

            # # Create figure (Modifié pour être plus allongé)
            fig, ax = plt.subplots(figsize=(10, 4))

            # # Plot Ours
            sns.lineplot(
                data=subset, x="NoiseLevel", y=our_col,
                label="Ours (Frangi)",
                color="#d62728", marker="o", linewidth=2.5, errorbar='sd', ax=ax
            )

            # # Plot CSD
            if csd_col in subset.columns and subset[csd_col].notna().any():
                sns.lineplot(
                    data=subset, x="NoiseLevel", y=csd_col,
                    label="CrackSegDiff",
                    color="#2ca02c", marker="s", linestyle="--", linewidth=2, errorbar='sd', ax=ax
                )

            # --- # Axes Configuration ---
            # # No title
            ax.set_title("")

            # # X Axis : 0 à 0.5, pas 0.1
            ax.set_xlim(0, 0.5)
            ax.xaxis.set_major_locator(ticker.MultipleLocator(0.1))
            ax.set_xlabel("") # # No axis label
            ax.set_xticklabels([]) # # No ticks (chiffres)

            # Axe Y
            ax.set_ylim(*ylims)
            ax.yaxis.set_major_locator(ticker.MultipleLocator(ystep))
            ax.set_ylabel("") # # No axis label
            ax.set_yticklabels([]) # # No ticks (chiffres)

            # # Dotted grid
            ax.grid(True, linestyle='--', alpha=0.7)

            # # Legend
            ax.legend(loc="best", frameon=True)

            plt.tight_layout()
            plt.show()

if 'df_noise' in locals():
    display_noise_results(df_noise)
else:
    print("Variable 'df_noise' is not defined.")